# R3 — LangChain basics

Reach the workspace chat model two ways (ChatDatabricks wrapper and OpenAI-compatible ChatOpenAI wrapper pointed at the workspace), and prove a chain can read a policy document and hand back a typed, validated Pydantic object (`WaitingPeriodFacts` via `PydanticOutputParser`) for both waiting-period documents (02 individual, then 03 group). Evidence is written to `evidence/r3_langchain.json`.

In [0]:
%python
# R3 Step 0: notebook-scoped LangChain install.
# langchain-databricks 0.1.2 metadata pins mlflow-skinny<3, so it MUST be installed with
# --no-deps — otherwise pip downgrades runtime core packages (mlflow, protobuf, pyarrow,
# packaging), which breaks MLflow/Spark in the session. Its non-core runtime deps
# (openai, tiktoken, databricks-vectorsearch, langsmith) are installed explicitly instead.
%pip install --quiet "langchain>=1.0" "langchain-core>=1.7.5,<2" "langchain-openai>=1.7" "databricks-vectorsearch>=0.30" tiktoken
%pip install --quiet --no-deps "langchain-databricks==0.1.2"

In [0]:
%python
# R3 Step 1: reach the chat model through the Databricks wrapper (ChatDatabricks).
# databricks-meta-llama-3-3-70b-instruct is a READY chat endpoint on this workspace.
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
WORKSPACE_HOST = w.config.host
MODEL = "databricks-meta-llama-3-3-70b-instruct"
print("workspace:", WORKSPACE_HOST)

from langchain_databricks import ChatDatabricks
chat_databricks = ChatDatabricks(endpoint=MODEL, temperature=0, max_tokens=2000)

reply = chat_databricks.invoke("In one sentence, name the two waiting periods a health policy sets.")
chat_databricks_reply = reply.content
print(type(reply).__name__, "->", chat_databricks_reply)

In [0]:
%python

from langchain_openai import ChatOpenAI

chat_openai = ChatOpenAI(
    model=MODEL,
    base_url=f"{WORKSPACE_HOST}/serving-endpoints",
    api_key=w.config.authenticate()["Authorization"].split(" ", 1)[1],
    temperature=0,
)

reply2 = chat_openai.invoke("In one sentence, name the two waiting periods a health policy sets.")
chat_openai_reply = reply2.content
print(type(reply2).__name__, "->", chat_openai_reply)

In [0]:
%python
# R3 Step 3: a chain that reads a policy document and hands back a typed, validated object.
# WaitingPeriodFacts, parsed with PydanticOutputParser, for BOTH waiting-period documents,
# 02_Waiting_Periods_Individual_Plans.md first, then 03_Waiting_Periods_Group_Plans.md.
# In langchain-core 1.x the parser lives in langchain_core.output_parsers.
import os
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

DOCS_DIR = "/Volumes/claims/raw/landing/genai/policy_docs"
WAITING_DOCS = ["02_Waiting_Periods_Individual_Plans.md", "03_Waiting_Periods_Group_Plans.md"]

class WaitingPeriodFacts(BaseModel):
    doc_name: str = Field(description="File name of the policy document, exactly, with no folder")
    plan_family: str = Field(description="Plan family named in the document's Applies to: line: individual or group")
    initial_waiting_days: int = Field(description="Initial waiting period in days for that plan family")
    ped_waiting_months: int = Field(description="Pre-existing disease (PED) waiting period in months for that plan family")

parser = PydanticOutputParser(pydantic_object=WaitingPeriodFacts)

prompt = PromptTemplate(
    template=(
        "Read the policy document and extract its waiting-period facts.\n"
        "The plan family MUST be the family named in the document's 'Applies to:' line "
        "(individual or group). Ignore any other plan mentioned in side sections.\n"
        "Report the initial waiting period in DAYS and the pre-existing disease (PED) "
        "waiting period in MONTHS, for THAT family.\n"
        "{format_instructions}\n\nDOCUMENT ({doc_name}):\n{doc_text}"
    ),
    input_variables=["doc_name", "doc_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
chain = prompt | chat_databricks | parser

facts_objects = []
for doc_name in WAITING_DOCS:
    with open(os.path.join(DOCS_DIR, doc_name)) as f:
        doc_text = f.read()
    facts = chain.invoke({"doc_name": doc_name, "doc_text": doc_text})
    facts_objects.append(facts)
    print(facts)

waiting_facts = [f.model_dump() for f in facts_objects]

In [0]:
%python
# R3 Step 4: evidence/r3_langchain.json — chat_databricks_reply, chat_openai_reply,
# waiting_facts (list of two objects, document 02 first then 03).
import json

evidence = {
    "chat_databricks_reply": chat_databricks_reply,
    "chat_openai_reply": chat_openai_reply,
    "waiting_facts": waiting_facts,
}

evidence_path = ("/Workspace/Users/deepaksinghal@cyntexa.com/"
                 "-DE---Scenario-based-Assignment-1/Gen AI - Paper 4 Scenario Based Assignment/"
                 "evidence/r3_langchain.json")
with open(evidence_path, "w") as f:
    json.dump(evidence, f, indent=2)

# verification: non-empty replies, exactly two facts, 02 before 03
assert evidence["chat_databricks_reply"].strip(), "chat_databricks_reply empty"
assert evidence["chat_openai_reply"].strip(), "chat_openai_reply empty"
assert len(waiting_facts) == 2, "expected two waiting_facts objects"
assert waiting_facts[0]["doc_name"] == "02_Waiting_Periods_Individual_Plans.md", waiting_facts[0]["doc_name"]
assert waiting_facts[1]["doc_name"] == "03_Waiting_Periods_Group_Plans.md", waiting_facts[1]["doc_name"]
assert all(isinstance(f["initial_waiting_days"], int) and isinstance(f["ped_waiting_months"], int) for f in waiting_facts)
print("evidence written:", evidence_path)
print(json.dumps(evidence, indent=2))